<a href="https://colab.research.google.com/github/susiegithub/NLP_CWK/blob/main/NLP_CWK_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main imports and code

In [6]:
# check which gpu we're using
!nvidia-smi

Wed Mar  4 11:48:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
%pip install tensorboardx
%pip install torch
%pip install "transformers<5" simpletransformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.8/330.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 124.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=d33f3567d16dd145af36588368c54bc9f73adf8f5b44b2071e99ad54eda5b679
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd74

In [8]:
import pandas as pd
import logging
import torch
from ast import literal_eval

In [9]:
# prepare logger
logging.basicConfig(level=logging.INFO)

transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# check gpu
cuda_available = torch.cuda.is_available()

print('Cuda available? ',cuda_available)

Cuda available?  True


In [10]:
if cuda_available:
  import tensorflow as tf
  # Get the GPU device name.
  device_name = tf.test.gpu_device_name()
  # The device name should look like the following:
  if device_name == '/device:GPU:0':
      print('Found GPU at: {}'.format(device_name))
  else:
      raise SystemError('GPU device not found')

Found GPU at: /device:GPU:0


# Load paragraph IDs

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
FILE_EXT = "/content/drive/MyDrive/NLP_CWK"
train_ids = pd.read_csv(f"{FILE_EXT}/train_semeval_parids-labels.csv")
dev_ids = pd.read_csv(f"{FILE_EXT}/dev_semeval_parids-labels.csv")

In [13]:
import ast

train_ids.par_id = train_ids.par_id.astype(str)
dev_ids.par_id = dev_ids.par_id.astype(str)

train_ids.label = train_ids.label.apply(ast.literal_eval)
dev_ids.label = dev_ids.label.apply(ast.literal_eval)

In [14]:
DATA_PATH = FILE_EXT + "/dontpatronizeme_pcl.tsv"
PCL_THRESHOLD = 2

df = pd.read_csv(
        DATA_PATH,
        sep="\t",
        skiprows=4,
        header=None,
        names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
        quoting=3,
    )
df["text"] = df["text"].astype(str)
df["par_id"] = df["par_id"].astype(str)
df["binary_label"] = (df["label"] >= PCL_THRESHOLD).astype(int)


In [15]:
df

,par_id,art_id,keyword,country_code,text,label,binary_label
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0
2,3,@@16584954,immigrant,ie,"""White House press secretary Sean Spicer said ...",0,0
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0
4,5,@@1494111,refugee,ca,""""""" Just like we received migrants fleeing El ...",0,0
...,...,...,...,...,...,...,...
10464,10465,@@14297363,women,lk,"""Sri Lankan norms and culture inhibit women fr...",1,0
10465,10466,@@70091353,vulnerable,ph,He added that the AFP will continue to bank on...,0,0
10466,10467,@@20282330,in-need,ng,""""""" She has one huge platform , and informatio...",3,1
10467,10468,@@16753236,hopeless,in,""""""" Anja Ringgren Loven I ca n't find a word t...",4,1




# Rebuild training set

In [16]:
rows = []

for idx in range(len(train_ids)):
  parid = train_ids.par_id[idx]
  row_data = df.loc[df.par_id == parid].iloc[0]  # get the first matching row

  rows.append({
      'par_id': parid,
      'community': row_data['keyword'],
      'text': row_data['text'],
      'label': row_data['label'],
      'binary_label': row_data['binary_label'],
      'keyword': row_data['keyword'],
      'category_labels': train_ids.label[idx],
  })


In [17]:
import random

In [18]:
original_train_df = pd.DataFrame(rows)

In [19]:
print(original_train_df['binary_label'].value_counts())
original_train_df

binary_label
0    7581
1     794
Name: count, dtype: int64


,par_id,community,text,label,binary_label,keyword,category_labels
0,4341,poor-families,"The scheme saw an estimated 150,000 children f...",4,1,poor-families,"[1, 0, 0, 1, 0, 0, 0]"
1,4136,homeless,Durban 's homeless communities reconciliation ...,2,1,homeless,"[0, 1, 0, 0, 0, 0, 0]"
2,10352,poor-families,The next immediate problem that cropped up was...,4,1,poor-families,"[1, 0, 0, 0, 0, 1, 0]"
3,8279,vulnerable,Far more important than the implications for t...,2,1,vulnerable,"[0, 0, 0, 1, 0, 0, 0]"
4,1164,poor-families,To strengthen child-sensitive social protectio...,4,1,poor-families,"[1, 0, 0, 1, 1, 1, 0]"
...,...,...,...,...,...,...,...
8370,8380,refugee,Rescue teams search for survivors on the rubbl...,0,0,refugee,"[0, 0, 0, 0, 0, 0, 0]"
8371,8381,hopeless,The launch of ' Happy Birthday ' took place la...,0,0,hopeless,"[0, 0, 0, 0, 0, 0, 0]"
8372,8382,homeless,"The unrest has left at least 20,000 people dea...",0,0,homeless,"[0, 0, 0, 0, 0, 0, 0]"
8373,8383,hopeless,You have to see it from my perspective . I may...,0,0,hopeless,"[0, 0, 0, 0, 0, 0, 0]"


# Data augmentation #


In [20]:
%pip install nlpaug nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 15.1 MB/s eta 0:00:00


In [22]:
import nlpaug.augmenter.word as naw
import pandas as pd
import random
import nltk
import re

nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [23]:
aug = naw.SynonymAug(aug_src='wordnet')

keywords = set()

for kw in original_train_df["keyword"]:
    for k in kw.split("-"):
        keywords.add(k)

def balance_dataframe(df, text_col, label_col, protected_words=None):
    class_counts = df[label_col].value_counts()
    majority_count = class_counts.max()

    augmented_rows = []

    # Build a placeholder map: keyword -> unique token unlikely to exist in text
    placeholder_map = {}
    reverse_map = {}

    if protected_words:
        for word in protected_words:
            placeholder = f"PROTECTEDKW{abs(hash(word)) % 100000}"
            placeholder_map[word.lower()] = placeholder
            reverse_map[placeholder] = word

    def mask_keywords(text):
        if not placeholder_map:
            return text

        words = text.split()
        masked = [placeholder_map.get(w.lower(), w) for w in words]
        return " ".join(masked)

    def unmask_keywords(text):
        if not reverse_map:
            return text

        words = text.split()
        restored = [reverse_map.get(w, w) for w in words]
        return " ".join(restored)

    for label, count in class_counts.items():
        if count < majority_count:
            minority_df = df[df[label_col] == label]
            needed = majority_count - count
            existing_texts = set(df[text_col].tolist())

            attempts = 0
            max_attempts = needed * 10

            while len(augmented_rows) < needed and attempts < max_attempts:
                row = minority_df.sample(1).iloc[0].copy()

                masked_text = mask_keywords(row[text_col])
                augmented_text = aug.augment(masked_text)[0]
                augmented_text = unmask_keywords(augmented_text)

                if augmented_text not in existing_texts:
                    row[text_col] = augmented_text
                    augmented_rows.append(row)
                    existing_texts.add(augmented_text)

                attempts += 1

            if len(augmented_rows) < needed:
                print(f"Warning: could only generate {len(augmented_rows)}/{needed} unique samples for class {label}")

    augmented_df = pd.DataFrame(augmented_rows)
    balanced_df = pd.concat([df, augmented_df], ignore_index=True).sample(frac=1).reset_index(drop=True)

    return balanced_df

# Pass the keywords set into the function
augmented_train_df = balance_dataframe(original_train_df, text_col="text", label_col="binary_label", protected_words=keywords)

In [24]:
augmented_train_df["binary_label"].value_counts()

,count
binary_label,
0,7581
1,7581


# Rebuild dev set

In [25]:
rows = []

for idx in range(len(dev_ids)):
  parid = dev_ids.par_id[idx]
  row_data = df.loc[df.par_id == parid].iloc[0]  # get the first matching row

  rows.append({
      'par_id': parid,
      'community': row_data['keyword'],
      'text': row_data['text'],
      'label': row_data['label'],
      'binary_label': row_data['binary_label'],
      'category_labels': dev_ids.label[idx],
  })


In [26]:
len(rows)

2094

In [27]:
dev_df = pd.DataFrame(rows)
dev_df

,par_id,community,text,label,binary_label,category_labels
0,4046,hopeless,We also know that they can benefit by receivin...,3,1,"[1, 0, 0, 1, 0, 0, 0]"
1,1279,refugee,Pope Francis washed and kissed the feet of Mus...,4,1,"[0, 1, 0, 0, 0, 0, 0]"
2,8330,refugee,Many refugees do n't want to be resettled anyw...,2,1,"[0, 0, 1, 0, 0, 0, 0]"
3,4063,in-need,"""Budding chefs , like """" Fred """" , """" Winston ...",4,1,"[1, 0, 0, 1, 1, 1, 0]"
4,4089,homeless,"""In a 90-degree view of his constituency , one...",3,1,"[1, 0, 0, 0, 0, 0, 0]"
...,...,...,...,...,...,...
2089,10462,homeless,"The sad spectacle , which occurred on Saturday...",0,0,"[0, 0, 0, 0, 0, 0, 0]"
2090,10463,refugee,""""""" The Pakistani police came to our house and...",0,0,"[0, 0, 0, 0, 0, 0, 0]"
2091,10464,disabled,"""When Marie O'Donoghue went looking for a spec...",0,0,"[0, 0, 0, 0, 0, 0, 0]"
2092,10465,women,"""Sri Lankan norms and culture inhibit women fr...",1,0,"[0, 0, 0, 0, 0, 0, 0]"


In [28]:
dev_df = dev_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(dev_df['binary_label'].value_counts())


binary_label
0    1895
1     199
Name: count, dtype: int64


# Read test set

In [ ]:
TEST_DATA_PATH = FILE_EXT + "/task4_test.tsv"

test_df = pd.read_csv(
        TEST_DATA_PATH,
        sep="\t",
        skiprows=0,
        header=None,
        names=["par_id", "art_id", "keyword", "country_code", "text"],
        quoting=3,
    )
test_df["text"] = test_df["text"].astype(str)
test_df["par_id"] = test_df["par_id"].astype(str)

In [ ]:
test_df = pd.DataFrame(rows)
test_df

In [ ]:
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Run inference on test dataset
# test_preds = predict(model, tokenizer, test_df.text.tolist())

# Run baseline model on augmented dataset

In [29]:
# # downsample negative instances
# pcldf = original_train_df[original_train_df.binary_label==1]
# npos = len(pcldf)

# training_set1 = pd.concat([pcldf, original_train_df[original_train_df.binary_label==0][:npos*2]])


In [35]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs


MANUAL_SEED = 42

baseline_model_args = ClassificationArgs(num_train_epochs=3,
                                      no_save=True,
                                      no_cache=True,
                                      overwrite_output_dir=True,
                                      manual_seed=MANUAL_SEED,
                                         )
baseline_model = ClassificationModel("roberta",
                                  'roberta-base',
                                  args = baseline_model_args,
                                  num_labels=2,
                                  # weight=[1.0, 5.0],
                                  use_cuda=cuda_available)
# train model
baseline_model.train_model(augmented_train_df[['text', 'binary_label']])

# run predictions on dev set
baseline_preds, baseline_outputs = baseline_model.predict(dev_df.text.tolist())

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:637: UserWarning: Dataframe headers not specified. Falling back to using column 0 as text and column 1 as labels.
  warnings.warn(


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:924: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 3:   0%|          | 0/1896 [00:00<?, ?it/s]

...

Running Epoch 2 of 3:   0%|          | 0/1896 [00:00<?, ?it/s]

Running Epoch 3 of 3:   0%|          | 0/1896 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

Predicting:   0%|          | 0/21 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [31]:
!pip install torchmetrics

In [36]:
from sklearn.metrics import f1_score, recall_score, precision_score

# Evaluate tuned model
y_true = dev_df['binary_label'].tolist()

y_pred = baseline_preds

f1 = f1_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")


F1 Score: 0.5047
Precision: 0.6780
Recall: 0.4020


# Hyper-parameter search on class weights (discarded)


In [ ]:
CLASS_WEIGHTING = false

In [ ]:
if CLASS_WEIGHTING:
    import gc

    weight_range = np.arange(5.0, 10.1, 1.0)  # try w_pos from 5 to 10
    best_threshold = None
    best_f1 = 0.0
    best_weight = None

    for w_pos in weight_range:
        print(f"Testing weight [1.0, {w_pos}]")

        model_args = ClassificationArgs(
            num_train_epochs=3,
            no_save=True,
            output_dir=f"output_w{w_pos}/",
            no_cache=True,
            overwrite_output_dir=True,
            evaluate_during_training=True,
            reprocess_input_data = True,
            # wandb_project =" Simple Sweep",
        )

        model = ClassificationModel(
            "roberta",
            "roberta-base",
            num_labels=2,
            weight=[1.0, w_pos],
            args=model_args,
            use_cuda=cuda_available,
            # sweep_config=wandb.config,
        )

        # train
        model.train_model(train_df=train_df[["text", "binary_label"]].rename(columns={"binary_label": "label"}),
                          args={"evaluate_during_training": True},
                          eval_data=val_df[["text", "binary_label"]].rename(columns={"binary_label": "label"}))

        # predict
        preds, outputs = model.predict(val_df.text.tolist())
        probs = softmax(torch.tensor(outputs).float().cpu().detach(), dim=1)[:, 1].numpy()

        # tune threshold for this weight
        local_best_f1 = 0.0
        local_best_thresh = None

        for t in np.arange(0.1, 0.6, 0.05):
            preds_thresh = (probs > t).astype(int)
            f1 = f1_score(val_df['binary_label'], preds_thresh)

            print(f"    t={t:.2f}, f1={f1:.4f}")

            if f1 > local_best_f1:
                local_best_f1 = f1
                local_best_thresh = t

        print(f"  Best threshold: {local_best_thresh}, F1: {local_best_f1}")

        # update global best
        if local_best_f1 > best_f1:
            best_f1 = local_best_f1
            best_threshold = local_best_thresh
            best_weight = [1.0, w_pos]

            # model.model.save_pretrained("best_weight_model")
            # model.tokenizer.save_pretrained("best_weight_model")
            print(f"  Saved new best model with F1: {best_f1}")

    print(f"\nOptimal weight: {best_weight}, threshold: {best_threshold}, F1: {best_f1}")


In [ ]:
if CLASS_WEIGHTING:
    # Reload the model
    saved_model = ClassificationModel(
        "roberta",
        "outputs/best_model/",  # path to saved model directory
        num_labels=2,
        use_cuda=cuda_available
    )

In [ ]:
if CLASS_WEIGHTING:
    # Evaluate tuned model
    y_true = dev_df['binary_label'].tolist()

    baseline_preds, baseline_outputs = saved_model.predict(dev_df.text.tolist())
    y_probs = softmax(torch.tensor(baseline_outputs), dim=1)
    y_probs = y_probs[:, 1] # positive class only
    y_pred = (y_probs > best_threshold).int().tolist()

    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)

    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")


# Define new model


In [37]:
import torch.nn as nn
from transformers import RobertaModel, RobertaTokenizer, get_linear_schedule_with_warmup

class MultiTaskPCLModel(nn.Module):
    def __init__(self, model_name='roberta-base', num_categories=7, dropout=0.1):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size  # 768 for base

        self.dropout = nn.Dropout(dropout)

        # Head 1: Binary PCL classification
        self.binary_head = nn.Linear(hidden_size, 2)

        # Head 2: Multi-label category classification
        self.category_head = nn.Linear(hidden_size, num_categories)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])

        binary_logits = self.binary_head(cls)        # (batch, 2)
        category_logits = self.category_head(cls)    # (batch, num_categories)

        return binary_logits, category_logits

In [38]:
from torch.utils.data import Dataset, DataLoader

class PCLDataset(Dataset):
    """
    Expects a DataFrame with columns:
      - 'text'         : raw text
      - 'binary_label' : 0 or 1 (non-PCL or PCL)
      - 'category_labels'   : list of ints, e.g. [0, 1, 0, 1, 0, 0, 0]
                         one entry per category (multi-hot encoded)
    """
    def __init__(self, df, tokenizer, max_length=100):
        self.texts = df['text'].tolist()
        self.binary_labels = df['binary_label'].tolist()
        self.cat_labels = df['category_labels'].tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'binary_label':   torch.tensor(self.binary_labels[idx], dtype=torch.long),
            'category_labels': torch.tensor(self.cat_labels[idx], dtype=torch.float)
        }



In [54]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchmetrics.functional import f1_score, precision, recall

def train_multitask(
    train_df,
    val_df = None,
    num_categories=7,
    num_epochs=10,
    batch_size=16,
    lr=2e-5,
    weight_decay=0.01,
    lambda_cat=0.3,
    save_path="best_model.pt",
    early_stopping_patience=3,
    early_stopping_min_delta=1e-4,
):

    device = torch.device("cuda" if cuda_available else "cpu")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

    train_dataset = PCLDataset(train_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    if val_df is not None:
      val_dataset = PCLDataset(val_df, tokenizer)
      val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = MultiTaskPCLModel(num_categories=num_categories).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs) # cosine annealing as small dataset, potentailly noisy

    binary_loss_fn = nn.CrossEntropyLoss()
    category_loss_fn = nn.BCEWithLogitsLoss()

    current_best_f1 = 0.
    epochs_without_improvement = 0

    # Training
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            binary_labels  = batch['binary_label'].to(device)
            cat_labels     = batch['category_labels'].to(device)

            binary_logits, cat_logits = model(input_ids, attention_mask)

            loss_binary = binary_loss_fn(binary_logits, binary_labels)
            loss_cats   = category_loss_fn(cat_logits, cat_labels)
            loss        = loss_binary + lambda_cat * loss_cats

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Validate and checkpoint
        if val_df is not None:
            model.eval()

            all_preds, all_labels = [], []

            with torch.no_grad():
                for batch in val_loader:
                    input_ids      = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)

                    binary_logits, _ = model(input_ids, attention_mask)

                    preds = torch.argmax(binary_logits, dim=1).cpu()
                    labels = batch['binary_label'].cpu()

                    all_preds.append(preds)
                    all_labels.append(labels)

            all_preds = torch.cat(all_preds)
            all_labels = torch.cat(all_labels)

            prec = precision(all_preds, all_labels, task="binary").item()
            rec  = recall(all_preds, all_labels, task="binary").item()
            f1 = f1_score(all_preds, all_labels, task="binary") # positive class F1
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val F1: {f1:.4f} | Val precision: {prec:.4f} | Val recall: {rec:.4f}", end="")

            # --- Early stopping logic ---
            if f1 > current_best_f1 + early_stopping_min_delta:
                current_best_f1 = f1
                epochs_without_improvement = 0

                torch.save({
                    'epoch':             epoch + 1,
                    'model_state_dict':  model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'f1':                current_best_f1,
                    'lambda_cat':        lambda_cat,
                }, save_path)
                print(f" Saved (best F1: {current_best_f1:.4f})")

            else:
                epochs_without_improvement += 1
                print(f" (no improvement for {epochs_without_improvement}/{early_stopping_patience} epochs)")

                if epochs_without_improvement >= early_stopping_patience:
                    print(f"\nEarly stopping triggered after epoch {epoch+1}.")
                    break

            # save if best so far
            if f1 > current_best_f1:
                current_best_f1 = f1
                torch.save({
                    'epoch':      epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'f1':         current_best_f1,
                    'lambda_cat': lambda_cat,
                }, save_path)
                print(f" Saved (best F1: {current_best_f1:.4f})")
        else:
            # No validation, just save the final epoch
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")
            if epoch == num_epochs - 1:
                torch.save({
                    'epoch':      epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'f1':         None,
                    'lambda_cat': lambda_cat,
                }, save_path)
                print(f"Final model saved to {save_path}")
            print()

        scheduler.step()

    # Load best checkpoint
    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"\nLoaded best checkpoint from epoch {checkpoint['epoch']} (F1: {checkpoint['f1']})")
    return model, tokenizer


In [40]:
def predict(model, tokenizer, texts, batch_size=16) -> torch.Tensor:
    device = next(model.parameters()).device
    model.eval()
    all_preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        encoding = tokenizer(
            batch_texts,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            binary_logits, _ = model(
                encoding['input_ids'],
                encoding['attention_mask']
            )

        preds = torch.argmax(binary_logits, dim=1)
        all_preds.append(preds.cpu())

    return torch.cat(all_preds)


# Train model on original dataset

In [55]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    original_train_df,
    test_size=0.1,
    random_state=42,
    stratify=original_train_df['binary_label']  # preserve class balance
)

print(train_df['binary_label'].value_counts())
print(len(train_df))
print(len(val_df))

binary_label
0    6822
1     715
Name: count, dtype: int64
7537
838


In [57]:
num_epochs = 10
# lambda_cat = 0.3 # weighting factor of how much the category loss contributes to the total loss
lr = 2e-5
weight_decay = 0.01

lambda_values = [0.1, 0.3, 0.5, 1.0]
best_lambda = None
best_f1 = 0.0

# Train on train/val split to tune hyper-parameters
for lmbda in lambda_values:
    print(f"\nTraining with lambda_cat = {lmbda}")

    model, tokenizer = train_multitask(
        train_df,
        val_df=val_df,
        lr=lr,
        weight_decay=weight_decay,
        lambda_cat=lmbda,
        save_path=f"original_dataset_model_lambda_{lmbda}.pt"
    )

    # After training, load best F1 from checkpoint
    checkpoint = torch.load(f"original_dataset_model_lambda_{lmbda}.pt", map_location="cpu")
    f1 = checkpoint["f1"]

    print(f"Validation F1: {f1}")

    if f1 > best_f1:
        best_f1 = f1
        best_lambda = lmbda

print(f"\nBest lambda_cat: {best_lambda} (F1: {best_f1})")


Training with lambda_cat = 0.1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.2878 | Val F1: 0.0494 | Val precision: 1.0000 | Val recall: 0.0253 Saved (best F1: 0.0494)
Epoch 2 | Loss: 0.1996 | Val F1: 0.2549 | Val precision: 0.5652 | Val recall: 0.1646 Saved (best F1: 0.2549)
Epoch 3 | Loss: 0.1357 | Val F1: 0.4333 | Val precision: 0.6341 | Val recall: 0.3291 Saved (best F1: 0.4333)
Epoch 4 | Loss: 0.0704 | Val F1: 0.4742 | Val precision: 0.4000 | Val recall: 0.5823 Saved (best F1: 0.4742)
Epoch 5 | Loss: 0.0311 | Val F1: 0.4308 | Val precision: 0.5490 | Val recall: 0.3544 (no improvement for 1/3 epochs)
Epoch 6 | Loss: 0.0147 | Val F1: 0.5000 | Val precision: 0.4624 | Val recall: 0.5443 Saved (best F1: 0.5000)
Epoch 7 | Loss: 0.0096 | Val F1: 0.4841 | Val precision: 0.4872 | Val recall: 0.4810 (no improvement for 1/3 epochs)
Epoch 8 | Loss: 0.0077 | Val F1: 0.4733 | Val precision: 0.5962 | Val recall: 0.3924 (no improvement for 2/3 epochs)
Epoch 9 | Loss: 0.0063 | Val F1: 0.4698 | Val precision: 0.5000 | Val recall: 0.4430 (no improvement for

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.3225 | Val F1: 0.4793 | Val precision: 0.6905 | Val recall: 0.3671 Saved (best F1: 0.4793)
Epoch 2 | Loss: 0.2276 | Val F1: 0.4923 | Val precision: 0.6275 | Val recall: 0.4051 Saved (best F1: 0.4923)
Epoch 3 | Loss: 0.1478 | Val F1: 0.3968 | Val precision: 0.5319 | Val recall: 0.3165 (no improvement for 1/3 epochs)
Epoch 4 | Loss: 0.0981 | Val F1: 0.4895 | Val precision: 0.5469 | Val recall: 0.4430 (no improvement for 2/3 epochs)
Epoch 5 | Loss: 0.0397 | Val F1: 0.4882 | Val precision: 0.6458 | Val recall: 0.3924 (no improvement for 3/3 epochs)

Early stopping triggered after epoch 5.

Loaded best checkpoint from epoch 2 (F1: 0.4923076927661896)
Validation F1: 0.4923076927661896

Training with lambda_cat = 0.5


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.3350 | Val F1: 0.0488 | Val precision: 0.6667 | Val recall: 0.0253 Saved (best F1: 0.0488)
Epoch 2 | Loss: 0.2311 | Val F1: 0.3366 | Val precision: 0.7727 | Val recall: 0.2152 Saved (best F1: 0.3366)
Epoch 3 | Loss: 0.1527 | Val F1: 0.5190 | Val precision: 0.5190 | Val recall: 0.5190 Saved (best F1: 0.5190)
Epoch 4 | Loss: 0.0947 | Val F1: 0.5068 | Val precision: 0.5522 | Val recall: 0.4684 (no improvement for 1/3 epochs)
Epoch 5 | Loss: 0.0531 | Val F1: 0.5000 | Val precision: 0.5065 | Val recall: 0.4937 (no improvement for 2/3 epochs)
Epoch 6 | Loss: 0.0378 | Val F1: 0.4895 | Val precision: 0.5469 | Val recall: 0.4430 (no improvement for 3/3 epochs)

Early stopping triggered after epoch 6.

Loaded best checkpoint from epoch 3 (F1: 0.5189873576164246)
Validation F1: 0.5189873576164246

Training with lambda_cat = 1.0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.3969 | Val F1: 0.2424 | Val precision: 0.6000 | Val recall: 0.1519 Saved (best F1: 0.2424)
Epoch 2 | Loss: 0.2839 | Val F1: 0.4310 | Val precision: 0.6757 | Val recall: 0.3165 Saved (best F1: 0.4310)
Epoch 3 | Loss: 0.1937 | Val F1: 0.5685 | Val precision: 0.4746 | Val recall: 0.7089 Saved (best F1: 0.5685)
Epoch 4 | Loss: 0.1156 | Val F1: 0.4706 | Val precision: 0.5614 | Val recall: 0.4051 (no improvement for 1/3 epochs)
Epoch 5 | Loss: 0.0770 | Val F1: 0.4744 | Val precision: 0.4805 | Val recall: 0.4684 (no improvement for 2/3 epochs)
Epoch 6 | Loss: 0.0541 | Val F1: 0.4490 | Val precision: 0.4853 | Val recall: 0.4177 (no improvement for 3/3 epochs)

Early stopping triggered after epoch 6.

Loaded best checkpoint from epoch 3 (F1: 0.5685279369354248)
Validation F1: 0.5685279369354248

Best lambda_cat: 1.0 (F1: 0.5685279369354248)


In [58]:
# Retrain on full dataset
final_model_on_original_dataset, tokenizer1 = train_multitask(
    train_df=original_train_df,
    val_df = None,
    lr = lr,
    weight_decay=weight_decay,
    num_epochs=3,
    lambda_cat = best_lambda,
    save_path="final_best_original_dataset_model.pt"
)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.3933

Epoch 2 | Loss: 0.2770

Epoch 3 | Loss: 0.1724
Final model saved to final_best_original_dataset_model.pt


Loaded best checkpoint from epoch 3 (F1: None)


# Inference on dev dataset

In [59]:
# Evaluate on dev dataset
dev_preds = predict(final_model_on_original_dataset, tokenizer1, dev_df.text.tolist())
dev_labels = torch.tensor(dev_df['binary_label'].values)
dev_f1_score = f1_score(dev_preds, dev_labels, task='binary')

print(f"Dev F1 score: {dev_f1_score}")

Dev F1 score: 0.5685785412788391


# Train the model on augmented data set #

In [60]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    original_train_df,
    test_size=0.1,
    random_state=42,
    stratify=original_train_df['binary_label']  # preserve class balance
)

print(train_df['binary_label'].value_counts())
print(len(train_df))
print(len(val_df))


binary_label
0    6822
1     715
Name: count, dtype: int64
7537
838


In [61]:
augmented_train_df = balance_dataframe(train_df, text_col="text", label_col="binary_label", protected_words=keywords)

In [ ]:
num_epochs = 10
# lambda_cat = 0.3 # weighting factor of how much the category loss contributes to the total loss
lr = 2e-5
weight_decay = 0.01

lambda_values = [0.1, 0.3, 0.5, 1.0]
best_lambda = None
best_f1 = 0.0

# Train on train/val split to tune hyper-parameters
for lmbda in lambda_values:
    print(f"\nTraining with lambda_cat = {lmbda}")

    model, tokenizer = train_multitask(
        augmented_train_df,
        val_df=val_df,
        lr=lr,
        weight_decay=weight_decay,
        lambda_cat=lmbda,
        save_path=f"augmented_dataset_model_lambda_{lmbda}.pt"
    )

    # After training, load best F1 from checkpoint
    checkpoint = torch.load(f"augmented_dataset_model_lambda_{lmbda}.pt", map_location="cpu")
    f1 = checkpoint["f1"]

    print(f"Validation F1: {f1}")

    if f1 > best_f1:
        best_f1 = f1
        best_lambda = lmbda

print(f"\nBest lambda_cat: {best_lambda} (F1: {best_f1})")


Training with lambda_cat = 0.1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.2111 | Val F1: 0.4889 | Val precision: 0.5893 | Val recall: 0.4177 Saved (best F1: 0.4889)
Epoch 2 | Loss: 0.1449 | Val F1: 0.4480 | Val precision: 0.6087 | Val recall: 0.3544 (no improvement for 1/3 epochs)


In [52]:
# Retrain on full dataset
full_train_df = pd.concat([augmented_train_df, val_df]).reset_index(drop=True)

final_model, tokenizer = train_multitask(
    train_df=full_train_df,
    val_df = None,
    lr = lr,
    weight_decay=weight_decay,
    num_epochs=num_epochs,
    lambda_cat = best_lambda,
)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 0.2783

Epoch 2 | Loss: 0.1702

Epoch 3 | Loss: 0.0903
Final model saved to best_model.pt


Loaded best checkpoint from epoch 3 (F1: None)


# Inference on dev dataset

In [53]:
# Evaluate on dev dataset
dev_preds = predict(final_model, tokenizer, dev_df.text.tolist())
dev_labels = torch.tensor(dev_df['binary_label'].values)
dev_f1_score = f1_score(dev_preds, dev_labels, task='binary')

print(f"Dev F1 score: {dev_f1_score}")

Dev F1 score: 0.5571847558021545


# Error Analysis


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(dev_labels, dev_preds)

sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['Non-PCL', 'PCL'],
            yticklabels=['Non-PCL', 'PCL'])

print(classification_report(dev_labels, dev_preds,
      target_names=['Non-PCL', 'PCL']))

In [ ]:
# Inspect misclassifications
dev_df['pred'] = dev_preds

false_positives = dev_df[(dev_df['label_binary']==0) & (dev_df['pred']==1)]
false_negatives = dev_df[(dev_df['label_binary']==1) & (dev_df['pred']==0)]

print("False Positives (predicted PCL, actually not):")
print(false_positives['text', "category_labels"].head(10).to_string())

print("\nFalse Negatives (missed PCL):")
print(false_negatives['text', "category_labels"].head(10).to_string())

In [ ]:
# Confidence analysis



In [ ]:
# Text length analysis
dev_df['text_length'] = dev_df['text'].apply(lambda x: len(x.split()))

correct   = dev_df[dev_df['correct'] == True]['text_length']
incorrect = dev_df[dev_df['correct'] == False]['text_length']

print(f"Avg length correct:   {correct.mean():.1f} words")
print(f"Avg length incorrect: {incorrect.mean():.1f} words")

## Prepare submission

In [ ]:
# helper function to save predictions to an output file
def labels2file(p, outf_path):
	with open(outf_path,'w') as outf:
		for pi in p:
			outf.write(','.join([str(k) for k in pi])+'\n')

In [ ]:
labels2file([[k] for k in dev_preds], 'dev.txt')
labels2file([[k] for k in test_preds], 'test.txt')

In [ ]:
!cat dev.txt | head -n 10

In [ ]:
!cat test.txt | head -n 10

In [ ]:
!zip submission.zip dev.txt test.txt